## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Defining Model

In [23]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0.1,
)

## Creating Agent

In [24]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [25]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

**Capital City:** **Selene‑Haven**

**Location:**  
Selene‑Haven is perched on the rim of the **Mare Imbrium** basin, atop the basaltic plateau known as **Luna Regia**. The city’s central plaza occupies a natural amphitheater carved into the ancient lava flow, giving it a panoramic view of the Earth rising over the horizon—a sight that can be seen from almost every balcony.

**Founding & Governance:**  
Founded in 2084 by the United Lunar Nations (ULN), Selene‑Haven serves as the political, cultural, and scientific heart of lunar civilization. It houses the **Lunar Council**, a bicameral assembly of representatives from the major lunar mining conglomerates, research colonies, and the increasingly influential **Luna‑Indigenous** communities who have lived on the Moon for three generations. The city’s mayor, currently **Governor Aria Kwan**, is elected directly by the citizens of the entire Moon, making Selene‑Haven the only capital in the Solar System with a truly planetary electorate.


## Image input

In [26]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.jpeg', multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpeg', description='Upload')

In [27]:
print(uploader.value)

({'name': 'images.jpeg', 'type': 'image/jpeg', 'size': 54960, 'content': <memory at 0x111bee200>, 'last_modified': datetime.datetime(2026, 7, 3, 6, 31, 48, 933000, tzinfo=datetime.timezone.utc)},)


In [28]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [30]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this image that I have shared"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

This image depicts a futuristic, extraterrestrial metropolis set on a desolate, rocky planetary surface—likely a moon or distant world. The city features sleek, towering structures with glowing warm-yellow lights, blending advanced technology with the harsh, cratered terrain. In the background, a massive planet (resembling Earth) looms in the star-studded void of space, its curved horizon glowing with atmospheric light. The scene evokes a sense of isolation and wonder, combining the stark beauty of an alien landscape with the vibrant, inhabited presence of a sci-fi civilization.


## Audio input

In [31]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.54it/s]


Done.


In [ ]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)